In [1]:

import pandas as pd
from wearable_affect.data import PROJECT_ROOT
from wearable_affect.evaluation import feature_columns
from wearable_affect.features import HEART_FEATURES
from wearable_affect.models import make_logreg, make_lightgbm
from wearable_affect.tracking import run_loso_experiment

processed = PROJECT_ROOT / "data" / "processed"
handcrafted = pd.read_parquet(processed / "features.parquet")
papagei = pd.read_parquet(processed / "papagei_embeddings.parquet")

keys = ["subject_id", "start_s", "target"]
non_heart = [c for c in feature_columns(handcrafted) if c not in HEART_FEATURES]

feature_sets = {
    "heart_handcrafted": handcrafted[keys + HEART_FEATURES],
    "heart_papagei": papagei,
    "all_handcrafted": handcrafted,
    "nonheart_handcrafted+papagei": handcrafted[keys + non_heart].merge(papagei, on=keys, validate="one_to_one"),
}
for name, df in feature_sets.items():
    print(f"{name:32s} {df.shape}")

heart_handcrafted                (1015, 7)
heart_papagei                    (1015, 515)
all_handcrafted                  (1015, 17)
nonheart_handcrafted+papagei     (1015, 525)


In [2]:
summary = []
for fs_name, df in feature_sets.items():
    for model_name, make_model in [("logreg", make_logreg), ("lightgbm", make_lightgbm)]:
        results = run_loso_experiment(
            f"{fs_name}-{model_name}", df, make_model,
            params={"window_s": 60, "step_s": 30, "threshold": 0.5,
                    "features": fs_name, "ppg_model": "papagei_s", "pooling": "mean"},
        )
        summary.append({"features": fs_name, "model": model_name,
                        **results[["macro_f1", "balanced_accuracy", "auroc"]].mean().round(3).to_dict()})

pd.DataFrame(summary)

,features,model,macro_f1,balanced_accuracy,auroc
0,heart_handcrafted,logreg,0.782,0.809,0.894
1,heart_handcrafted,lightgbm,0.795,0.808,0.887
2,heart_papagei,logreg,0.773,0.774,0.865
3,heart_papagei,lightgbm,0.784,0.789,0.882
4,all_handcrafted,logreg,0.831,0.852,0.966
5,all_handcrafted,lightgbm,0.850,0.866,0.976
6,nonheart_handcrafted+papagei,logreg,0.813,0.825,0.915
7,nonheart_handcrafted+papagei,lightgbm,0.880,0.883,0.954


In [ ]:
import mlflow
from wearable_affect.tracking import TRACKING_URI

mlflow.set_tracking_uri(TRACKING_URI)
runs = mlflow.search_runs(experiment_names=["wesad-stress"])

def per_subject(run_name: str) -> pd.DataFrame:
    """Load the per-subject results saved with a run (the most recent run with that name)."""
    run_id = runs.loc[runs["tags.mlflow.runName"] == run_name, "run_id"].iloc[0]
    path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path="per_subject.csv")
    return pd.read_csv(path).set_index("subject_id")

hand = per_subject("all_handcrafted-lightgbm")
pap = per_subject("nonheart_handcrafted+papagei-lightgbm")

comparison = pd.DataFrame({
    "f1_hand": hand["macro_f1"], "f1_papagei": pap["macro_f1"],
    "auroc_hand": hand["auroc"], "auroc_papagei": pap["auroc"],
})
comparison["f1_diff"] = comparison["f1_papagei"] - comparison["f1_hand"]
comparison["auroc_diff"] = comparison["auroc_papagei"] - comparison["auroc_hand"]
print(comparison.round(3))
print(f"PaPaGei better F1 for {(comparison['f1_diff'] > 0).sum()} of {len(comparison)}, "
      f"better AUROC for {(comparison['auroc_diff'] > 0).sum()}")

            f1_hand  f1_papagei  auroc_hand  auroc_papagei  f1_diff  \
subject_id                                                            
S2            0.852       0.852       0.994          0.990    0.000   
S3            0.735       0.759       0.884          0.898    0.024   
S4            0.962       1.000       1.000          1.000    0.038   
S5            0.944       0.882       0.984          0.981   -0.062   
S6            0.826       0.763       0.922          0.928   -0.063   
S7            0.868       0.820       1.000          0.956   -0.048   
S8            0.907       0.983       1.000          1.000    0.075   
S9            0.876       0.899       1.000          0.985    0.023   
S10           0.736       0.966       1.000          1.000    0.229   
S11           0.890       0.920       0.987          0.998    0.030   
S13           1.000       1.000       1.000          1.000    0.000   
S14           0.415       0.668       0.909          0.683    0.253   
S15   

: 